# Workshop: How Short-Form Content Recommenders Work 📱

* **Location**: EdinburghAI Workshops
* **Credits**: Adapted from the EdinburghAI workshop created by Conor O'Shea.
* **Prerequisites**: None. No coding experience or math background required. You only need a web browser and an AI chat tool (such as Gemini, ChatGPT, or Claude).

---

# What is this workshop about? 🤔

When you scroll through TikTok or Instagram Reels, the app seems to read your mind. If you pause on two study tips and quickly skip past three dance clips, your feed shifts almost instantly.

How does the algorithm make that decision in under a tenth of a second?

In 2024, Meta published a landmark paper called *Actions Speak Louder Than Words*. They introduced an architecture called the **Hierarchical Sequential Transduction Unit (HSTU)**. Meta's key insight was simple: recommendation is not just about what topics you like in general. It is about reading your scrolling behavior as an ongoing sequence:

```text
[Video Shown] -> [Your Action] -> [Video Shown] -> [Your Action] -> [Candidate Video] -> [Predict Action]
```

Today, you will not write any code. Instead, you will act as the recommendation architect. You will use a live reels app to record your own scrolling sequence, and then you will use a web-based AI (like Gemini) as the computational brain to score videos and predict what you will want to see next.

---

# Section 1: The Anatomy of a Scrolling History 📜

Traditional recommendation systems treated users like static library cards: "Alex likes cats (80%), likes comedy (70%), dislikes politics (90%)."

Modern sequential recommenders do not do this. They know human interest changes from minute to minute. The order of events tells a story.

Suppose Alex is scrolling through reels:

1. Alex sees a cooking tutorial for creamy pasta. Alex watches all 30 seconds and taps "Like."
2. Alex sees a video about chess grandmasters. Alex watches for 2 seconds and immediately skips.
3. Alex sees a student meme about exam panic. Alex watches the entire clip, smiles, and opens the comments section.

At any moment, the algorithm breaks reality into three distinct roles:

* **History**: Everything that happened in the past (the videos shown and how the user reacted).
* **Candidate Input**: A new reel from the catalogue that the app is considering showing right now. The app knows its title, sound, and duration, but does not know if the user will like it.
* **Prediction Target**: The outcome the app wants to predict (for example: "What is the probability the user skips this candidate in under 3 seconds?").

## Task 1.0: Sorting the Timeline 🔢

Here are three events from Alex's session, but they are scrambled out of order:

* **Event A (16:04:10)**: Shown an acoustic guitar cover. Watched 40 seconds, tapped "Like."
* **Event B (16:01:00)**: Shown a stand-up comedy sketch. Watched 3 seconds, skipped.
* **Event C (16:02:30)**: Shown a student study playlist. Watched 45 seconds, saved to collection.

**Questions to answer in your notes**:
1. What is the chronological order of these events?
2. Look at the transition from Event B to Events C and A. What shift in Alex's mood or setting took place between 16:01 and 16:04?

## Task 1.1: History, Candidate, or Target? 🔮

Suppose the algorithm is evaluating a new candidate reel: **"5 Hidden Study Spots in Edinburgh."**

Classify each of the following pieces of information as **History**, **Candidate Input**, or **Prediction Target**:

1. Alex liked an acoustic guitar video two minutes ago.
2. The Edinburgh study spots reel is 15 seconds long.
3. Will Alex watch at least 80% of this Edinburgh study spots reel?
4. Alex quickly skipped a comedy sketch at the start of the session.
5. The Edinburgh study spots reel is tagged with `#edinburgh`, `#study`, and `#university`.
6. Will Alex tap the "Share" button on this reel?

---

# Section 2: The Video-Action Sequence 🎵

Language models like Gemini read text as a sequence of words or tokens. In the same way, sequential recommenders read your scroll as an alternating chain of **Content Tokens** and **Action Tokens**:

```text
[Video Token] -> [Action Token] -> [Video Token] -> [Action Token] ...
```

Why keep the video and the action separate?

* `[Comedy Video]` followed by `[Skip]` teaches the algorithm something completely different than `[Comedy Video]` followed by `[Share]`.
* An action token (like "watched 100% and liked") only makes sense when attached to the specific video token that came right before it.

### How the Ranking Score is Calculated

Instead of asking a vague question like "Is this reel good?", production recommenders predict probabilities for specific actions and combine them using a weighted score:

$$\text{Final Score} = (1.5 \times P(\text{Watch})) + (2.0 \times P(\text{Like})) + (2.5 \times P(\text{Share})) - (3.0 \times P(\text{Skip}))$$

Notice the negative weight on skip. If a reel has a high chance of annoying the user or causing a quick swipe-away, its score plummets.

---

# Section 3: Live Doomscrolling 😵‍💫

Now let us gather your real interaction data.

We have set up a live web application containing a catalogue of approved Instagram reels.

### Step 1: Open the App
Open the workshop app in a new browser tab:
[Open the EdinburghAI Reels App](https://edi-ai-app-reels-app-g9hix2dhtyjvntnkkvnzcd.streamlit.app/)

### Step 2: Scroll Naturally
* Watch whatever catches your interest.
* Tap Like when you enjoy a clip.
* Skip quickly when you find a clip boring or uninteresting.
* Scroll through at least **10 to 15 reels** so the session has enough sequence history to reveal your preferences.

### Step 3: Copy Your Scroll Log
* When you finish scrolling, look for the export section or copy button in the app.
* Click **Copy** to copy your session log (this will be a snippet of JSON text containing your watch times, reel IDs, and likes).
* Keep this copied text on your clipboard for the next step!

---

# Section 4: Turning an LLM into Your Recommendation Engine 🤖

Now you will turn a web AI (such as Gemini, ChatGPT, or Claude) into an HSTU sequential recommendation engine.

The prompt below gives the AI:
1. The sequential reasoning rules (weighting recent videos over older ones, penalizing skips).
2. The mathematical formula for ranking.
3. A candidate catalogue of real reels from the workshop app.

### The Master Recommender Prompt

Copy the entire prompt box below (including the long catalogue of videos), open your AI chat tool, and paste it in:

```text
You are the EdinburghAI Sequential Recommendation Engine, modeled after Meta's HSTU architecture.

Your objective is to examine a user's real-time scrolling history, identify their sequential momentum, score candidate reels from our catalogue, and output the top 10 recommended reel IDs.

HOW TO ANALYZE THE SEQUENCE:
1. Read the user history as alternating pairs: [Reel Content] -> [Action Taken].
2. Pay closest attention to recent interactions. An interaction that happened 30 seconds ago carries more weight than one from 5 minutes ago.
3. Positive signals: High watch ratio (watched most of duration), likes, and shares.
4. Negative signals: Quick skips (watched 3 seconds or less), negative feedback, or swiping away early.

SCORING FORMULA:
For each candidate, estimate three probabilities based on user sequence momentum:
- P(Watch): Probability user watches at least 75% of duration (0.0 to 1.0)
- P(Like): Probability user taps Like (0.0 to 1.0)
- P(Skip): Probability user skips within 3 seconds (0.0 to 1.0)

Calculate Final Score = (1.5 * P(Watch)) + (2.0 * P(Like)) - (3.0 * P(Skip))

INSTRUCTIONS FOR OUTPUT:
When I paste my interaction JSON log below:
1. Provide a 2-sentence summary diagnosing my taste and current mood momentum.
2. Filter out any reel IDs that I already watched in my history.
3. Rank the top 10 recommended candidate reels by Final Score.
4. Present the final 10 reel IDs as a clean Python-style list of strings, exactly like this:
['id1', 'id2', 'id3', 'id4', 'id5', 'id6', 'id7', 'id8', 'id9', 'id10']
5. Explain in two sentences why your #1 pick best matched my recent sequence.

THE CANDIDATE CATALOGUE (CSV FORMAT):
reel_id,category,content_format,title,tags
C0uLmD9PRCE,finance_money_meme,meme,joking about getting the ick from a girl because she doesnt particpate in tax evasion,"humor, satire, finance, taxevasion"
C2P0yXdugGf,work_relatable_brainrot,meme,joking that you only work hard so you can please they shareholders meme,"humor, corporate, larp, shareholdervalue"
C2gRdz5LI25,internship_meme,meme,joking that your internship managers will think highly of you,"internship, humor, relatable, satire"
C5rNmy6uSaQ,student_relatable_meme,meme,watching marketing/management students not know what they are doing,"finance, university, humor"
C6T54MMOrAT,pop_culture_brainrot,meme,baby gronk talks about hugging livvy dunne,"babygronk, livvydunne, brainrot"
C9dtcSJx9CT,relationship_dating_meme,meme,manifesting relationship meme,"dating, humor, relatable"
CuIOwZyxHz4,pop_culture_brainrot,meme,livvy dunne talks on podcast about baby gronk,"babygronk, livvydunne, brainrot"
DEibBnMIpRC,finance_money_meme,meme,joking about spending lots of money as soon as your first year analyst bonus hits,"finance, humor, relatable, financebro"
DFEBdqgReB9,pop_culture_brainrot,meme,baby gronk rizzes up livvy dunne,"brainrot, babygronk, livvydunne"
DIX2BNzJYJQ,finance_money_meme,meme,joking about forcing your future son into going into quantitative finance,"quantfinance, finance, humor, satire"
DIwHhT1RR4A,finance_money_meme,meme,joking about owning mcdonalds after buying one share in mcdonalds,"stocks, finance, larp, humor, batman"
DKLkkR3taYR,relationship_dating_meme,meme,trying to choose better parnters but the nice ones text too corny,"singlelife, humor, dating, relatable"
DKulsdPNUkN,finance_money_meme,meme,joking about feelng high brow after using pointless finance jargon,"finance, humor, larp, relatable"
DM_k1gtywFV,pop_culture_brainrot,meme,theo von asks if rizzler owns a scale,"brainrot, humor, theovon, rizzler"
DMn7qJ0B15X,finance_money_meme,meme,when your portfolio is up by $6.73 so you start forgetting your day 1s meme,"satire, humor, finance, stocks"
DRcyBNQiTaJ,finance_money_meme,meme,joking about losing all morallity after getting a high paid private equity job,"humor, finance, privateequity, immoral"
DRsLs-UDlVk,finance_money_meme,meme,when your finance job is atcual work instead of just pretending your patrick bateman,"humor, brainrot, finance, patrickbateman, americanpsycho"
DSRPOYckYe7,relationship_dating_meme,meme,pov: you start dating someone attractive and you feel people waiting for you to mess up,"relatable, humor, dating, hypergamy, pov"
DS_beibEnRp,tech_interview_advice,instructional,3 things i studied to get a big tech internship,"technical interview, tech, faang"
DU7V-N9EbBf,pop_culture_brainrot,meme,47 cruise missiles vs janitor,"brainrot, johnkiriakou, cia"
DUgsQg_juKu,finance_money_meme,meme,joking about being upset when you actually have to do work at your finance job,"humor, finance, patrickbateman, americanpsycho, larp"
DVHGysMDfC1,finance_money_meme,meme,joking about using finance jargon to make your situation sound much better,"finance, humor, satire"
DVwbSYUDcw0,relationship_dating_meme,meme,crush isnt replying so you just go to bed instead meme,"dating, relatable, humor"
DW6TIGVjZCS,finance_money_meme,meme,joking about dropouting out of school after receiving a single $0.01 divedend,"finance, larp, humor, stocks, trading"
DW9Jh58jS05,combat_sports_brainrot,meme,forgetting everything that coach just said meme,"mma, wrestling, relatable, humor, joerogan"
DWUGDmNjTT6,pop_culture_brainrot,meme,baby gronk plays reaction game,"brainrot, babygronk"
DWXynSyuvGM,tech_interview_advice,instructional,tech projects to stand out from the crowd advice,"technical interview, tech"
DWhmX5uCcYY,student_relatable_meme,meme,chatgpt is giving you the wrong answers so you actually have to think for yourself,"chatgpt, ai, brainrot, humor, relatable, satire"
DWkUSI0jqkP,tech_interview_advice,instructional,3 technical interview tips,"technical interview, tech"
DWt3H5ACp1J,relationship_dating_meme,meme,nervous about meeting her family for the first time meme,"dating, humor, relatable, stadiumpulse"
DWxNOYKDNIy,work_relatable_brainrot,meme,pov: you have a question that you know youve probably asked 3 times already in work,"humor, relatable, work culture"
DXX8TZ6zB2D,pop_culture_brainrot,meme,someone asks for your name but you dont know it cause the chatgpt servers are down,"chatgpt, ai, brainrot, humor"
DXZeGEFER_r,work_relatable_brainrot,meme,having nothing to add to the meeting so you just use loads of coporate jargon,"satire, relatable, humor, corporate"
DXaSHy6uQ_U,work_relatable_brainrot,meme,struggling to find 3 fun facts about yourself meme,"corporate, humor, relatable"
DXh4nbPTEtT,student_relatable_meme,meme,professor mustve had a bad day before grading your essay meme,"assignments, university, humor, relatable"
DXpkzvoDMjV,relationship_dating_meme,meme,acting up accidentally when meeting your girfriends parents for the first time,"humor, dating, brainrot"
DXsSSVggVjW,student_relatable_meme,meme,joking about receiving so many job application rejections,"jobmarket, relatable, humor"
DYaHI2TBteT,combat_sports_brainrot,meme,getting beaten by everyone at the mma club because youre bad at it,"relatable, humor, mma, starwars, anakinskywalker"
DZ24jttCWV9,work_relatable_brainrot,meme,being too scared to walk away from your desk when youre only a juniot dev,"tech, relatable, humor"
DZ5LDTKJCv7,relationship_dating_meme,meme,ragebaiting girlfriend and it works meme,"dating, humor, relatable, ragebait"
DZ7Ws_MyUIg,work_relatable_brainrot,meme,pov: microsoft teams is your sleep paralysis demon,"microsoft, humor, relatable, corporate, workfromhome, microsoftteams"
DZ9eXfXtDbI,work_relatable_brainrot,meme,trying not to look greedy by snacking in front of manager meme,"humor, relatable, work culture"
DZELwuZC2Tq,pop_culture_brainrot,meme,dying old people regret not scrolling more reels satire,"reels, humor, brainrot"
DZKyyKzoTt_,pop_culture_brainrot,meme,john kiriakou answers commonly asked questions from fans on podcast,"johnkiriakou, cia, brainrot"
DZShRXxxmjV,pop_culture_brainrot,meme,john kiriakou loves angrybirds,"johnkiriakou, cia, humor"
DZZNSErNxdP,internship_meme,meme,intern on meeting with people making 10x their salary,"internship, humor, salary gap"
DZ_ALa1RjUc,work_relatable_brainrot,meme,when your boss asks you to share your screen but you have tabs open,"corporate, microsoftteams, humor, relatable, boss"
DZa0JvaBsH8,work_relatable_brainrot,meme,rizzler meme boss asking about task taking too long,"brainrot, boss, corporate, rizzler"
DZa79r1CoS0,work_relatable_brainrot,meme,taking a 5 hour break after 30 mins of productive work meme,"humor, brainrot, work culture, relatable"
DZabzvVTzkb,pop_culture_brainrot,meme,thats my quant clip from the big short film,"humor, tech, finance, quantfinance"
DZbJ84govrM,work_relatable_brainrot,meme,missing the people you used to work with cause they moved to a new job,"humor, coworker, relatable"
DZgzieShGpB,combat_sports_brainrot,meme,sparring beginners is low cortisol meme,"boxing, sparring, humor, relatable, lowcortisol"
DZhugVSDiNf,tech_brainrot,meme,that feeling when it worsk in production like it did in localhost,"humor, relatable, tech"
DZibns1ucXM,tech_interview_advice,instructional,best github repos for projects,"technical interview, tech, github"
DZjHIjVxR7p,student_relatable_meme,meme,joking about how tough it is to get a job in the current graduate job market,"interstellar, humor, relatable, university, jobmarket"
DZtNO_2pWn-,work_relatable_brainrot,meme,youre about to fall asleep at work but you look over and see your coworker wired,"humor, relatable, coworker"
Da2GygNyvBI,student_relatable_meme,meme,chatgpt graduation speech meme,"ai, assignments, university, chatgpt"
Da4Gk7DhuZ6,combat_sports_brainrot,meme,forgetting day 1s after winning 1 amateur fight meme,"satire, humor, boxing, mma"
Da5fLTQgTcZ,work_relatable_brainrot,meme,when youre working but youre coworker asks you to go on a walk around the office,"relatable, humor, brainrot, work culture, coworker, office"
Da5squGvcTz,internship_meme,meme,internship referral from bro meme,"referral, internship, humor"
Da6IUd8yaoS,finance_money_meme,meme,joking about niche economics concepts not being useful in day to day life,"finance, humor"
DaBW9BMy-Ve,combat_sports_brainrot,meme,coach being glad that your strip fell off your belt cause youre bad at bjj,"bjj, humor, relatable"
DaBtyQ5NUk4,tech_interview_advice,instructional,resources for breaking into quant,"quantfinance, technical interview, internship"
DaDuu9hqf1v,work_relatable_brainrot,meme,pov: teams messages wont let you enjoy your friday,"microsoftteams, workfromhome, humor, relatable"
DaFlNzoSZPr,combat_sports_brainrot,meme,looking for easy sparring partners meme,"boxing, humor, sarcasm"
DaKpOP0oZC0,relationship_dating_meme,meme,when the girl at work lowk cute so you ask her where she works meme,"dating, humor, relatable, brainrot, fumble"
DaKzMCUBKK6,work_relatable_brainrot,meme,joking that the onlyy person who will ever be obsessed with you is your manager,"satire, humor, relatable, dating, boss"
DaORNY_K-1O,work_relatable_brainrot,meme,remote workers getting up 1 minute before work starts meme,"workfromhome, humor, relatable, corporate"
DaPc06fSOA7,tech_interview_advice,instructional,how i got invited to google apple and microsoft,"technical interview, tech, microsoft, faang, google, apple"
DaQA9A4B9ys,work_relatable_brainrot,meme,when you accidentally send a private teams message in the meeting chat,"work culture, relatable, microsoftteams, humor"
DaQJaniMF2X,combat_sports_brainrot,meme,coach playing white girl music when youre hard sparring your friend,"relatable, humor, mma, boxing"
DaQvrUlBLjO,tech_brainrot,meme,finding out interview didnt go well after you thought youd aced it,"humor, relatable, technical interview"
DaSCqOtvN5p,work_relatable_brainrot,meme,leadership cooking up another reorg that nobody asked for meme,"relatable, humor, corporate"
DaSfMIOtE_6,combat_sports_brainrot,meme,starting to behave on social media after your taekwondo coach starts following you,"humor, relatable, taekwondo"
DaT2Ev0Ttx5,work_relatable_brainrot,meme,i dont want to climb the ladder i want to play on my phone meme,"humor, satire, corporate, bluecollar"
DaV6ApthtTB,work_relatable_brainrot,meme,coworker ending you reels during working hours meme,"humor, relatable, corporate, doomscroll, reels"
DaYsd9SB5hJ,work_relatable_brainrot,meme,opening tiktoks when on authenticator meme,"relatable, humor, tiktok, authenticator, microsoft"
Da_AQX4MZR2,work_relatable_brainrot,meme,being asked at work on a monday if your weekend was relaxing but it wasnt,"relatable, humor, work culture, mondayblues"
Daa_lShMpOT,work_relatable_brainrot,meme,joking about not being able to focus on work because of instagram reels addiction,"humor, reels, brainrot, doomscroll"
DadhKKoP_8Y,student_relatable_meme,meme,meme about struggling to get a job in the current job market,"jobmarket, humor, relatable"
DaegBUwhk77,pop_culture_brainrot,meme,when you like a movie but you find out it has low ratings on imbd,"humor, relatable, fittingin, imbd, movie"
DagexOkpkn7,combat_sports_brainrot,meme,mma commentators giving odd personal facts about fighters meme,"mma, brainrot, humor"
DahaTxGz-3w,finance_money_meme,meme,joking about switching up on your friends after getting a yearly $0.38 s&p 500 dividend,"humor, finance, trading, stocks"
DamH7OSpUZt,work_relatable_brainrot,meme,bored at work so actually try working meme,"humor, relatable, brainrot"
DaniSfrN4dP,work_relatable_brainrot,meme,your manager follows you on instagram so you start behaving well online,"boss, humor"
DaqLI6ohrny,combat_sports_brainrot,meme,conor mcgregor makes a creative excuse as to why he was throwing bottles into a crowd,"humor, mma, conormcgregor"
DardeAOSyJh,tech_brainrot,meme,struggling to think of git commit message meme,"tech, humor, relatable, git, github"
DasKKwJoSKg,pop_culture_brainrot,meme,joke about netflix showing search suggestions for movies that arent available,"humor, relatable, netflix"
DawWHI5JAZH,tech_interview_advice,instructional,best ways to land interviews at faang companies,"faang, tech, technical interview"
DawZwm1Oq5_,tech_interview_advice,instructional,technical interview advice,"tech, technical interview, swe"
DawuH0EOR_8,tech_brainrot,meme,when the manager politely asks me to review the pr instead of scrolling on my phone,"tech, relatable, humor, brainrot, boss"
DazT5rKzg_F,work_relatable_brainrot,meme,manager asking what youve been up to but its the mbappe special,"humor, brainrot, mbappespecial, football"
Db0HA3TpF6L,work_relatable_brainrot,meme,joking about dressing up fancy for your job where you just do nothing but scroll reels,"humor, relatable, reels, brainrot, doomscroll"
Db3gBMlhZ5X,work_relatable_brainrot,meme,skit about coworker who has no filter and will just say whats on their mind,"humor, startup, tech, satire"
Db41buBJKXZ,relationship_dating_meme,meme,when the girl at work lowk cute so you ask where she works meme,"singlelife, humor, relatable, fumble, dating"
Db69FG1p1lG,work_relatable_brainrot,meme,joking about rushing to leave the office as soon as you can cause you dont like your job,"humor, work culture, relatable"
Db6KPXaILW8,tech_brainrot,meme,when the hacker tells me my street aress but i lowkey already knew it,"tech, brainrot, humor, satire"
Db7KlAIopO3,tech_brainrot,meme,pov: when you spend 7 hours deubbing only to find out the issue was really silly,"tech, relatable, humor, debugging"
Db7Qn9RBTIf,work_relatable_brainrot,meme,when youve been watching reels in the bathroom for 2 hours and suddenly remember youre at work,"work culture, reels, humor, doomscroll, brainrot, breakingbad"
Db8q_pBMCqz,work_relatable_brainrot,meme,trying to watch reels but have to work instead meme,"reels, work culture, humor, brainrot, doomscroll"
Db9nNmMoFVZ,pop_culture_brainrot,meme,people who enjoy drinking matcha must enjoy eating grass meme,"performativemale, humor, matcha"
DbBbLwioM0u,internship_meme,meme,friends losing all time to hangout after they land an internship,"internship, tech, humor, relatable"
DbBzaG1DOyk,tech_brainrot,meme,trying to code a project without using ai but failing meme,"brainrot, ai, humor, relatable"
DbD4KmTBAxA,pop_culture_brainrot,meme,how i feel after 20 girls asked me to go out (I was in the wrong bathroom),"brainrot, humor, dating, jamesbond"
DbEu6iVtI2m,student_relatable_meme,meme,when your taking verbal abuse from higher ups in your summer job,"humor, relatable, summerjob, summerbreak"
DbFPhc6t8PQ,relationship_dating_meme,meme,finding girlsfriends girl drama kinda interesting meme,"dating, humor, relatable"
DbHVO5NvkRS,tech_brainrot,meme,tech people struggling to explain what their startup actually does meme,"ai, ai hype, startup, humor, satire, stanford"
DbLA-7sPFBf,pop_culture_brainrot,meme,when your phone dies on a night out and youre very far away from home,"humor, theodyssey, relatable"
DbLijOLRiYH,tech_brainrot,meme,enterprise java developers overengineering code meme,"relatable, humor, tech, java"
DbLoPNoIlmH,pop_culture_brainrot,meme,finding out the money laundering course you signed up for was actually about preventing money laundering,"humor"
DbMHAT-N-3Y,tech_brainrot,meme,on call swe engineer always getting paged meme,"swe, humor, relatable, tech"
DbMecVZhT-j,tech_interview_advice,instructional,best way to study for techincal interviews,"technical interview, tech, swe"
DbQF3eOTL-r,internship_meme,meme,forgetting day 1s after landing unpaid engineering internship meme,"satire, humor, brainrot"
DbQQJYRSL5B,combat_sports_brainrot,meme,training kicks being important for nonsense reason meme,"brainrot, humor"
DbRBkexRg2H,combat_sports_brainrot,meme,realising that the the gloves werent the problem and your just a bad fighter,"relatable, humor, boxing"
DbRGRFdAD5E,work_relatable_brainrot,meme,pov: youre a 20 year old teenager working in corporate so you keep wanting to message in modern slang,"humor, relatable, work culture, corporate, genz"
DbTpjWLzibs,combat_sports_brainrot,meme,upper belts switching up from being so nice when they actually spar you,"relatable, humor, bjj"
DbUOuTlzNFj,tech_brainrot,meme,pov: you and your tech leading trying to understand the code written by ai,"tech, humor, ai, relatable, bigbangtheory"
DbV3W85vMra,pop_culture_brainrot,meme,tsunami coming but chatgpt uses water,"chatgpt, ai, brainrot"
DbV4FEEMjy3,tech_brainrot,meme,pov scrum masters realizing nobody updated their tickets meme,"tech, humor, satire, agile"
DbV6m9EuO-l,internship_meme,meme,wondering what people did at internships before reels existed meme,"internship, doomscroll, brainrot, reels, humor"
DbWDwxAN-KK,tech_interview_advice,meme,bait and switch tech interview advice,"humor, technical interview, leetcode"
DbWzVMOTH5F,combat_sports_brainrot,meme,not undersanding flirting so thinking that a girl whos into you actually wants to fight you,"mma, boxing, humor, singlelife"
DbYluyKMM17,work_relatable_brainrot,meme,businesses changing requirements after development finishes,"tech, humor, relatable"
Db_-buozD1i,student_relatable_meme,meme,art students and computer science students both struggling to find jobs after graduation,"university, humor, relatable, jobmarket"
Db_1I2LvfxT,work_relatable_brainrot,meme,when you get distracted from instagram reels and actually do some work meme,"humor, relatable, brainrot, doomscroll, reels"
DbbF65kIFzW,work_relatable_brainrot,meme,guilt tripping boss for making you hop on a 4pm teams call with them by showing them a baby photo of you,"microsoftteams, corporate, humor, relatable, childhood, wholesome, workfromhome"
DbcEmDYuB_2,tech_brainrot,meme,how i imagine network protocols would act like if they were human meme,"tech, brainrot, humor"
DbcaOuQsQxA,internship_meme,meme,forgetting bro aftyer landing an unpaid internship,"humor, internship"
DbgGgUsgkNU,tech_brainrot,meme,joking about not really reviewing the code that claude ai generated and just approving it anyway,"ai, humor, relatable, tech"
DbmHQlIpt_I,work_relatable_brainrot,meme,skit about a coworker at a startup who has no filter and will just say whats on their mind,"humor, startup, skit"
DbmIOajyrLg,tech_interview_advice,instructional,how to have claude make your resume unrejectable meme,"tech, claude, technical interview, ai"
Dbn-HsXPpza,work_relatable_brainrot,meme,joking about how interns perk up when higher ups mention full time offers,"internship, humor, relatable"
DboF53HpvvJ,work_relatable_brainrot,meme,how i feel by ragebaiting my coworkers by saying damn i wasnt even born yet,"genz, humor, relatable, coworker"
DbqO16RRz7n,finance_money_meme,meme,using chatgpt as your financial advisor but it fails epically meme,"humor, chatgpt, ai, ai hype, finance, trading, stocks"
DbqaK9RMQco,combat_sports_brainrot,meme,whe she wants me to make the first move but little does she know im a counter fighter,"mma, dating, humor, relatable"
Dbr7RPWuLj1,work_relatable_brainrot,meme,blue collar workers wouldn't understand meme,"corporate, humor, work culture, relatable"
Dbsw7U0RAep,work_relatable_brainrot,meme,feeling productive after sending 3 emails in a row without checking instagram,"reels, humor, doomscroll, brainrot"
DbwafFlpXyu,work_relatable_brainrot,meme,skit about inappropriate startup groupchat getting leaked to hr,"tech, humor, hr, startup"
Dc-QNRCsG3z,pop_culture_brainrot,meme,joking about that feeling when you didnt send a meme to your friend cause you thought they wouldnt find it funny,"humor, relatable, doomscroll, brainrot"
Dc031TgJjvx,tech_brainrot,meme,when you finally get a gf but you cant get rid of the thought that shes only with you because you own a bambu lab printer,"humor, brainrot, dating, niche, 3dprinter"
Dc0z8acAuS_,tech_interview_advice,instructional,system design tips for avoiding recommendation loops,"technical interview, system design"
Dc1JGDkxXXu,work_relatable_brainrot,meme,pov: you joined the teams call too early and instantly regret it as they are all talking about nonsense,"humor, relatable, microsoftteams, corporate"
Dc1eAQfyIyO,relationship_dating_meme,meme,being socially awkward at social events but still being confused that you cant find a partner,"introvert, humor, relatable, ryangosling"
Dc1ilpNM3-5,student_relatable_meme,meme,feeling old because everyone during freshers is a lot younger than you when youre in upper university years,"university, relatable, humor"
Dc1nXQzIqZd,combat_sports_brainrot,meme,bringing gun to sparring meme,"humor, mma"
Dc1om9_PyfO,work_relatable_brainrot,meme,no one is more on social media than someone who works a 9-5 office job,"corporate, humor, relatable, brainrot, doomscroll"
Dc27HNXgAdc,tech_brainrot,meme,claude code watching you manually write code after you hit your daily claue usage limit but your code quality is terrible,"claude, ai, relatable, tech, humor"
Dc2FLa5tSrV,combat_sports_brainrot,meme,hot take: jiu jitsu has better grappling than kickboxing meme,"humor, bjj, boxing"
Dc2ntjRS7Kv,relationship_dating_meme,meme,expecting date after liking a girls instagram story,"humor, introvert, satire"
Dc30akeOh0C,pop_culture_brainrot,meme,spanish waiter says lo siento but it makes you think of a nieche pop culture meme,"brainrot, humor, nieche"
Dc3vsjaQFQM,tech_brainrot,meme,having updates to give in standup for once meme,"corporate, tech, humor, relatable, standup, meeting"
Dc3w0UWy3JJ,student_relatable_meme,meme,when bro says we are cooked but im actually not cause ive been studying in secret,"humor, relatable, exam, mbappe"
Dc4ODTboida,gaming_meme,meme,ps5 is more important that girlfriend breaking up with you meme,"playstation, gaming, humor, satire, ps5"
Dc4iprYixFb,pop_culture_brainrot,meme,want energy drink but too late in day,"humor, energydrink"
Dc51NaYsMm9,student_relatable_meme,meme,expectation vs reality of doing a group project at university,"university, humor, relatable, groupproject"
Dc5ZQVwpM3R,internship_meme,meme,pov: youre an intern in a random meeting with people making 10x your salary but theyre speaking in buzzwords,"internship, salary gap, relatable, humor, corporate"
Dc6CbK2zzan,combat_sports_brainrot,meme,when you have 13 seconds left on the exam so you lowk turn into prime conor mcgregor,"mma, humor, exam"
Dc6OVelx2E3,combat_sports_brainrot,meme,dog doing bjj with owner,"humor, bjj, wholesome, dog, pet"
Dc6UWYFsFXk,combat_sports_brainrot,meme,trying to beat your day 1s when coach is filming the bjj sparring session,"bjj, humor, relatable, satire"
Dc6oCOFpn0h,pop_culture_brainrot,meme,my mum when she realises i made a valid point in an argument so she doesnt want to talk anymore,"family, humor, relatable, topgear"
Dc7BdP5xmBQ,pop_culture_brainrot,meme,taking pride in useless hobbies meme,"humor, relatable"
Dc7uB1dMKOJ,pop_culture_brainrot,meme,quick reels break but forgets pram meme,"reels, brainrot, doomscroll, humor"
Dc9lD4-sRlF,pop_culture_brainrot,meme,when youre walking the wrong way but you dont want to lose aura by randomly turning around,"relatable, humor, brainrot, aura, auraloss"
DcAEpbQhvmM,tech_brainrot,meme,coding peter question appears in interview meme,"technical interview, tech, humor, brainrot, familyguy"
DcBBxy-zGdc,student_relatable_meme,meme,computer science majors being addicted to valorant while failing classes meme,"tech, humor, brainrot, relatable, gaming, discord, valorant"
DcES1mFkrEa,pop_culture_brainrot,meme,joking about being unhygenic,"satire, humor, brainrot, dating"
DcF0GEvNnxO,student_relatable_meme,meme,feels annoying to have the knowledge to do something but zero experience actually doing it,"jobmarket, humor, relatable"
DcG6v9FR9A6,tech_brainrot,meme,signs youre larping as a software engineer meme,"swe, humor, relatable, larp, tech"
DcGmGimudhD,work_relatable_brainrot,meme,boss asking why it took 37 hours to do a 2 hour job meme,"rizzler, boss, humor"
DcHMUgfBqWw,work_relatable_brainrot,meme,when youre trying to watch some reels but your job keeps getting in the way so you get annoyed,"theodyssey, humor, relatable"
DcHcMU-yrj-,work_relatable_brainrot,meme,intern coming back as staff after internship meme,"humor, relatable, internship, returnoffer"
DcK5BI4IVJm,pop_culture_brainrot,meme,making an insane internet reference and they dont get it so i have to find the original,"humor, relatable, brainrot, niche"
DcKEubEgTfY,pop_culture_brainrot,meme,laughing at your phone being on 67% then realizing your brain is rotted,"brainrot, humor, relatable, 67"
DcL5dwxte6x,pop_culture_brainrot,meme,wishing that your cat had a phone so you could send relatable reels to it,"reels, humor, relatable, cat"
DcLxR8ZtSIc,tech_brainrot,meme,pov: if other jobs interviewed like software engineering,"relatable, technical interview, humor, swe"
DcMH8ZBB6sz,student_relatable_meme,meme,talking to chatgpt lots before the night of an exam meme,"chatgpt, assignments, ai, university, relatable"
DcMQWDTpyDU,work_relatable_brainrot,meme,working from home but have to turn on camera suddenly meme,"corporate, humor, relatable, workfromhome"
DcMwpiiRoGr,relationship_dating_meme,meme,girl ghosting you right after you tell someone about her meme,"relatable, humor, dating, singlelife"
DcMyPETOer7,work_relatable_brainrot,meme,ragebaiting my older wiser slower coworkers by starting off a sentence with when im your age,"humor, relatable, ragebait"
DcNO_9buvqP,pop_culture_brainrot,meme,saying that only sleeping for 3 hours a night doesnt affect you but it clearly makes you hallucinate,"relatable, humor, brainrot, poorsleepquality"
DcOGKYUq6ir,work_relatable_brainrot,meme,the longer you work in it the more youll understand that your sucess is directly correlated with your ability to handle stress,"tech, humor, relatable"
DcOj2pCMQhu,student_relatable_meme,meme,there's two types of people when it comes to shopping for moving to university,"university, relatable, humor"
DcPRqhcunmg,tech_brainrot,meme,seeing if ai can help me talk to stranges but it fails in a funny way,"ai, ai hype, humor"
DcPiAFPpkJT,work_relatable_brainrot,meme,skit about a socially awkward coworker who never wants to hang out after work at a startup,"humor, relatable, skit, startup"
DcQ5OxsKEPH,work_relatable_brainrot,meme,when you realize you can not skip work like how you used to skip classes meme,"humor, relatable, brainrot"
DcQ6XdrNYI4,tech_brainrot,meme,pov: if software engineers interviewed like normal jobs meme,"tech, technical interview, humor, relatable"
DcR8W_ztt7z,tech_brainrot,meme,remembering the stackoverflow days before ai meme,"humor, relatable, tech, ai, stackoverflow"
DcTRFd2oHHU,internship_meme,meme,getting an intern to make your life easier but they have no work to do so they actually get in the way,"internship, humor, relatable"
DcWK2Fyopwt,pop_culture_brainrot,meme,pov: its my morning coffee and im reacting to the reels bro sent me like they're work emails,"humor, relatable, reels"
DcWyntXPwju,pop_culture_brainrot,meme,watch me scroll reels before bed meme,"reels, humor, brainrot, doomscroll"
DcY7QYWMXZz,tech_interview_advice,instructional,github repos with projects that may help you get a quant job at a top firm,"quantfinance, finance, tech, github, janestreet, citadel"
DcYIlZKvu6m,work_relatable_brainrot,meme,ranking corporate jargon phrases meme,"humor, corporate, relatable, brainrot"
DcZ0Y1Vpvd1,work_relatable_brainrot,meme,getting a call from a recruiter about random jobs on indeed after applying meme,"humor, relatable, indeed"
Dc_6UWqhjJi,pop_culture_brainrot,meme,bro sending freaky texts to his gf but you pretend not to see them,"humor, relatable, dating"
Dc_9PbeMUhh,pop_culture_brainrot,meme,joking about baggy jeans being able to carry lots of booze,"humor, party, booze"
Dc_R6xZtkEC,pop_culture_brainrot,meme,your sleep paralysis demon is a man in linen,"humor, relatable, posh, uk"
Dc_SxqIOA6z,tech_brainrot,meme,claude being more reliable than a girlfriend meme,"brainrot, relatable, humor, dating, claude, ai"
Dc__UNmJvJt,student_relatable_meme,meme,job portal tier list showcasing that nepotism is the best way to get a job,"jobmarket, humor, nepotism"
Dc_k2XXKLZa,combat_sports_brainrot,meme,tom aspinall jon jones callout meme,"mma, brainrot, humor, tomaspinall, jonjones"
Dc_l1LfxBTp,tech_brainrot,meme,claude prompt is too scary to send alone meme,"tech, claude, ai, relatable, humor, brainrot"
Dc_sNI4v1LZ,combat_sports_brainrot,meme,using cheap strategy but doesnt work in a real fight meme,"dating, humor, brainrot, bjj"
Dc_tIWNTZs3,relationship_dating_meme,meme,when you lwk admiring her but she catches you staring at her so you gotta lock back in,"dating, relatable"
Dc_tOXqodxf,pop_culture_brainrot,meme,telling bro youll be at his house in 5 mins when you havent even left your house yet,"humor, relatable"
Dc_ztnToBbw,work_relatable_brainrot,meme,getting home from work and realising you have to do it all over again tomorrow,"humor, relatable, inbetweeners"
DcaFI9dOigF,internship_meme,meme,forgetting day 1s after return offer meme,"internship, humor"
DcbM2lQNb7d,work_relatable_brainrot,meme,work flying you out to a terrible hotel but pretending its amazing meme,"humor, satire, sarcasm, relatable, corporate"
DcblqyfpJR_,tech_brainrot,meme,joking about suspicious software engineering terminology,"humor, swe"
DcceNm-AdES,pop_culture_brainrot,meme,joking about how anyone over the age of 25 only different emojis to genz,"humor, relatable, genz"
DccofTzMcvD,work_relatable_brainrot,meme,joking about finding love on linkedin instead of somewhere conventional meme,"humor, brainrot, linkedin"
Dcdp_EXOIV-,combat_sports_brainrot,meme,not knowing what youre doing whenever you spar meme,"rizzler, brainrot, humor, mma, relatable"
DceDTX7xi_e,tech_brainrot,meme,realizing that ai does all your coding work for you now and you dont really do anything anymore,"humor, tech, claude, claudecode, ai, ai hype, relatable"
DceyWyCp4f7,tech_brainrot,meme,skit about making an anonymous complaint to hr at a startup,"humor, tech, startup, skit, hr"
DcfQZP-RxyG,student_relatable_meme,meme,i hate ai folks when assignment due in an hour,"ai, assignments, university"
DcgYuqlzzXO,relationship_dating_meme,meme,when the convo gets dry with the 10/10 so u snitch on ur day one,"humor, relatable"
Dcgz5kaT-1d,combat_sports_brainrot,meme,coaching having you do weird warmup drills for bjj,"bjj, relatable, humor"
DchYXrJu9dF,work_relatable_brainrot,meme,joking about how it professionals get annoyed when you ask for help without submitting a ticket,"humor, relatable, corporate"
DchanFBtQI-,finance_money_meme,meme,pov: the company you own $5.43 of announces share buybacks and raises guidance on the same day,"finance, humor, relatable, stocks, trading"
DcigcJaNxno,work_relatable_brainrot,meme,joking about wanting to use genz slang in the coporate workplace but you cant,"genz, relatable, humor"
DcjBlC1quqn,work_relatable_brainrot,meme,job is driving me crazy but actually watching reels meme,"reels, brainrot, satire, humor, relatable"
DckUWgxsOXm,tech_interview_advice,instructional,how to quickly learn for system design interview,"system design, technical interview"
Dcl2f5mKFWL,pop_culture_brainrot,meme,grandson reading book instead of scrolling reels,"brainrot, reels, humor"
DcleUvgK6f3,combat_sports_brainrot,meme,my ceo neurosurgeon wife watching me show off my boxing combo at the family reunion,"boxing, brainrot, dating, humor, hypergamy"
Dclmsr2CTiR,tech_brainrot,meme,charleston ai launch video,"ai, ai hype, humor"
DcoVKpKsLKo,student_relatable_meme,meme,that one friend who cant comprehend that not everybody has a plan for after university,"humor, skit, relatable, university, jobmarket"
DcqTcDys3w6,tech_brainrot,meme,when bro asks if a for loop is gonna handle 10 billion database records in real time,"humor, tech"
Dcql9PbBAiJ,student_relatable_meme,meme,joking that even watching reels on mute is more entertaining than some lectures,"reels, humor, relatable, brainrot, doomscroll, university, lecture"
Dcr-a5nyfQf,combat_sports_brainrot,meme,getting annoyed at coach showing everyone how to counter your favourite techinques,"bjj, relatable, humor"
DcrDmfeJi6z,student_relatable_meme,meme,not being able to study because your cat is in the way,"wholesome, cat, pet, humor, relatable, study, university"
DcrlNwHzsUc,student_relatable_meme,meme,joking about how if youre late to university class registration there are often no desirable courses left,"university, humor, relatable"
DcrnZ2MxTLp,relationship_dating_meme,meme,always finding it hard to process that you actually like someone romantically,"dating, singlelife, humor, relatable"
DctBvsav5mo,work_relatable_brainrot,meme,joking about project managers wanting unreasonable amounts of work done in short time frames,"tech, humor, relatable"
DcttLcUvdN9,student_relatable_meme,meme,joking about students who go to the library on the first week of term even though no work has been released,"university, humor, larp"
Dcu9appz7Qt,pop_culture_brainrot,meme,Using a random reel as torch at 3am meme,"reels, humor, relatable"
DcvGBBIupbt,tech_interview_advice,instructional,how to prepare for tech ai infrastructure interview advice,"technical interview, tech, ai"
DcvLxW4qyL1,pop_culture_brainrot,meme,joking about being jealous of someone who can false asleep within 5 minutes of laying down when you have insomnia,"starwars, anakinskywalker, humor, relatable, insomnia"
Dcvc2HVIv-A,tech_brainrot,meme,guaranteeing that test cases pass but fails in production meme,"tech, testcases"
Dcw141dOBnd,tech_brainrot,meme,when youre stuck on an it issue so you summon the bald bearded coworker,"humor, relatable, tech"
DcwISV_JnIh,relationship_dating_meme,meme,too scared to ask girl out low cortisol meme,"humor, relatable"
DcwnRqvlKFM,tech_brainrot,meme,charleston ai progress update video,"ai, ai hype, humor"
Dcx-SIhCzNV,work_relatable_brainrot,meme,being the youngest in the office and feeling like a child meme,"humor, brainrot, relatable, work culture"
Dcx0QZJNaL-,pop_culture_brainrot,meme,when im in couples therapy with my wife and she says i wont even say hello to the neighbour,"dating, brainrot, humor, gaming, helloneighbour, niche"
DcxNm15tGob,pop_culture_brainrot,meme,when i accidentally pack my drink that exploes when it detects an owl meme,"brainrot, humor"
DcxSXlfApLQ,relationship_dating_meme,meme,trying to making a decision with your partner but you both want the other person to decide,"humor, dating, relatable, peoplepleaser"
Dcxj1nCC_G3,work_relatable_brainrot,meme,youre having a rough day at work then you turn to your coworker and they have it even worse,"relatable, humor, work culture, coworker"
DcyB-e9MNhc,student_relatable_meme,meme,the two types of friends who make career plans for after graduation meme,"humor, relatable, university"
DcyY5zJusJj,tech_interview_advice,instructional,advice on how many leetcode problems you should do for interview readiness,"leetcode, technical interview"
DcyhPQgzx8q,relationship_dating_meme,meme,youre on a date and she says that she saw a reel you liked on instagram but some of the stuff you like is morally questionable so youre nervous,"reels, humor, relatable"
DczHcl3Cg2U,tech_brainrot,meme,spending 20 hours automating a task that takes 20 mins to manually meme,"tech, humor, relatable"
DczJ19ToNnP,work_relatable_brainrot,meme,manager telling you about the big plans they have for you at the company but they dont know you actually want to leave soon,"relatable, humor"
DdA1ReQTe5Q,gaming_meme,meme,talking to bro about how much you both claim to hate the videogame youre currently playing but you keep playing it anyway,"gaming, humor, relatable"
DdA9LQBJI3h,student_relatable_meme,meme,joking about distracting your friend in the library when theyre focused and youre not,"humor, university, study, relatable, procrastination, lockedin"
DdACEv9htB2,work_relatable_brainrot,meme,coporate guys always looking for bars and pubs meme,"humor, finance, corporate, relatable"
DdADEQAEYtt,gaming_meme,meme,still using the ps4 given how old it is in 2026 meme,"humor, relatable, gaming, playstation, ps4"
DdASq35Tnjd,combat_sports_brainrot,meme,friend asking to try a new throw on you and then unexpectely launching you meme,"humor, bjj, relatable"
DdAVAJdvVka,pop_culture_brainrot,meme,having a bad day and then see a funny photo that your little cousin posted,"relatable, humor, family"
DdAWiCIgGeZ,student_relatable_meme,meme,youre playing games in class but then realize everyone else is taking notes so you feel guilty,"university, humor, relatable"
DdAYyalRe7b,student_relatable_meme,meme,guys pls dont ask me to hang out i have assignments to do and i will say yes meme,"assignments, humor, relatable, university"
DdA_2rOO99s,pop_culture_brainrot,meme,Me coming out of my cave because a lady ive never seen before wants to see how much ive grown,"relatable, humor, family"
DdAa13TORq6,work_relatable_brainrot,meme,missing a coworker you got on with is worse than a breakup meme,"humor, relatable, dating, coworker"
DdAjdM7Po0p,pop_culture_brainrot,meme,pov: the snack cart on the plane is coming so you need to let them know youre interested,"relatable, humor, holiday, aeroplane"
DdAm6ZMN86p,pop_culture_brainrot,meme,when you havent seen bro in so long you start looking at old photos like youre a divorced couple,"humor, relatable"
DdAm6wvuAWP,pop_culture_brainrot,meme,when the laundry trip finally makes it out the groupchat meme,"brainrot, satire, humor, university"
DdAmeaesRCv,pop_culture_brainrot,meme,pov: when youre about to leave the house so you charge your phone from a weak 20% to a strong 20% before you leave (it makes no difference but you still do it anyway),"relatable, humor, brainrot"
DdAo1EFtlzG,student_relatable_meme,meme,deciding to study in bed but actually falling asleep meme,"university, humor, study"
DdAoUpCsAc7,pop_culture_brainrot,meme,me if treating women right suddenly became ilegal performative meme,"satire, humor, performativemale"
DdAp0NRhPS4,combat_sports_brainrot,meme,some mma fighers have unfit looking physiques meme,"mma, humor, relatable"
DdAwPiCNwX5,pop_culture_brainrot,meme,trying to become a spinjitzu master when you were 7 years old relatable meme,"relatable, humor, legoninjago"
DdAz6k2grOy,work_relatable_brainrot,meme,pov: you just love all the in-person collaboration happening at the office (sarcastic as theres actually no one there meme),"relatable, workfromhome, humor, sarcasm"
DdB1XeDMmN5,pop_culture_brainrot,meme,me in the gym after buying keratin instead of creatine,"gym, humor"
DdB4IH-I3B6,pop_culture_brainrot,meme,meme about that one friend who never wants to wash any dishes so finds workarounds,"humor, relatable, uk"
DdB9_Kjzqpr,pop_culture_brainrot,meme,boat in hot tub meme,"humor, brainrot"
DdBKOAVD2oe,pop_culture_brainrot,meme,throwback to forging parents signature on 3rd grade reading log meme,"humor, relatable, wolfofwallstreet, childhood"
DdBSKkYoiAC,pop_culture_brainrot,meme,dog standing in the way of the television and owner getting annoyed meme,"wholesome, humor, dog, pet"
DdBSdcmTnP_,work_relatable_brainrot,meme,pov: devs hearing sales promise features that arent built yet and hating it as they have a big backlong,"relatable, humor, tech"
DdBT2FxMz1B,tech_brainrot,meme,guys i just started vibe coding when does this claude code come in when actually on scratch,"tech, humor, ai, claudecode, claude, scratch"
DdBaC-FSP8e,pop_culture_brainrot,meme,me when my mym asks me who Im hanging out with tonight meme. theyre not a good influence,"humor, relatable, trump"
DdBeuQKOhcG,relationship_dating_meme,meme,people claiming to have never been that into their ex but struggling to see her with another man,"relatable, humor, detroitbecomehuman"
DdBh_-cglNv,combat_sports_brainrot,meme,danawhite lying in interview meme,"humor, brainrot, mma, danawhite"
DdBjLWSg-m8,work_relatable_brainrot,meme,construction workers messing about instead of working even though project behind schedule,"humor, relatable, bluecollar"
DdBltGqB8aE,pop_culture_brainrot,meme,cat reading science textbook meme,"study, humor, cat"
DdBmAP0sZZH,pop_culture_brainrot,meme,my favourite thing to do is watch my own stories after i postm them cause its like danganother banger what,"brainrot, humor"
DdBmxJ2TrIv,student_relatable_meme,meme,when you highkey didnt study for your exam so you eat the gum under the table to get the knowledge from the previous owners,"exam, humor, relatable"
DdBnSEyAyqZ,combat_sports_brainrot,meme,losing a boxng fight so joking about using an illegal move,"boxing, humor"
DdBn_19ykJK,student_relatable_meme,meme,chatgpt being cool in 2022 vs taking away your job in 2026,"ai, ai hype, chatgpt, university, relatable, humor"
DdBvwgtqEKO,pop_culture_brainrot,meme,me finally deciding after 6 months to open bros 99+ reels he sent me and theyre low-key peak,"reels, humor, relatable, brainrot, interstellar"
DdBziM6TRt3,pop_culture_brainrot,meme,pov: me wasting my entire day because i have to do something at 5pm meme,"relatable, procrastination, humor"
DdC0VnWqdTX,pop_culture_brainrot,meme,pov: soldier in the trojan war who has a friend named troy so talking about going inside troy sounds suspicious,"brainrot, humor, theodyssey"
DdCJLL2sqNa,student_relatable_meme,meme,joking about it being needllessly hard to connect to university wifi for the first time,"university, humor, relatable, eduroam"
DdCNUXDo9T9,student_relatable_meme,meme,trying to go with the flow of life but you need to be locked in because the cs job market is cooked,"ai, study, jobmarket, humor, relatable, tech"
DdCRRM6sG8q,work_relatable_brainrot,meme,when youre helping your boss find out who made the mistake and you find out it was you,"relatable, humor, boss"
DdCTzxQJOX_,student_relatable_meme,meme,when youve been rejected from so many jobs that you can now spot rejection emails really quickly,"internship, jobmarket, relatable, humor"
DdCVIc5uKG7,internship_meme,meme,missing standup desk from internship when back at college after internship finishes,"relatable, humor, brainrot"
DdCd5JQC4mz,work_relatable_brainrot,meme,when your stressed at work but you dont smoke so you just blow bubbles instead,"humor, satire, relatable"
DdCfS_RD-eP,work_relatable_brainrot,meme,new hire spends 80+m tokens on ai slop,"ai, humor, relatable, tokens"
DdCgIXFBfz1,pop_culture_brainrot,meme,trying to watch sports on television on 2026 but they are all on different streaming services and channels,"humor, relatable, sports"
DdCgisJhnaV,tech_interview_advice,instructional,offering advice for the recently released big tech summer internship applications,"technical interview, tech, faang"
DdCr0dDpFUk,work_relatable_brainrot,meme,pov: asking your boss for paid time off in 2026 and its hard to get meme,"tech, humor, relatable"
```

### Step 4: Paste Your History into the LLM

After sending the master prompt, paste your copied JSON log directly into the chat.

Your message can be as simple as:

```text
Here is my scrolling log from the app:

[PASTE YOUR COPIED JSON LOG HERE]

Please predict and rank my top 10 reels according to the instructions.
```

---

# Section 5: Testing Your Recommendations in the Live App 🎬

Did the algorithm predict what you actually want to watch?

1. Copy the list of 10 reel IDs generated by the LLM (for example: `['C5rNmy6uSaQ', 'Dcql9PbBAiJ', ...]`).
2. Switch back to the [EdinburghAI Reels App](https://edi-ai-app-reels-app-g9hix2dhtyjvntnkkvnzcd.streamlit.app/).
3. Find the **Custom Feed / Recommended Reels** input box in the app.
4. Paste your list of recommended IDs into the box and load your custom feed.
5. Watch the reels recommended by the model!

### Reflection Questions:
* How accurately did the list match your taste?
* If you skipped corporate memes early in your session, did the LLM successfully filter out corporate content from the top recommendations?
* Did the recommendation feel personalized to your last few interactions?

---

# Section 6: How Engineers Evaluate Recommenders (Hit@10) 🎯

In a school quiz, accuracy is binary: you got the answer right or wrong.

In short-form recommendation, catalogues have millions of items. If an algorithm had to predict the single exact video you watch next out of 100,000 choices, it would be marked wrong almost every time, even if its guess was fantastic.

Recommender engineers use ranking metrics instead.

### What is Hit@K?

* **K** is the size of the shortlist shown to the user (usually 10 reels).
* **Hit@10**: Did at least one reel that the user genuinely enjoyed (watched in full or liked) land in the top 10 recommendations?
  * If yes: Score = 1 (Hit)
  * If no: Score = 0 (Miss)

When tested over thousands of users, the average Hit@10 score tells engineers how well the system populates feeds with appealing content.

### The Power of Being Better Than Chance

In a catalogue of 30,000 videos, pure random chance gives an expected Hit@10 of about **0.033%**.

If a sequential model achieves a Hit@10 of **3.5%**, that might look low compared to a classroom test, but it is actually **over 100 times better than random guessing**.

---

# Summary Checklist 📝

You have explored the foundational ideas behind modern sequential recommendation:

1. **Sequences tell stories**: Chronological order reveals changing mood in ways static profiles cannot.
2. **Content and actions alternate**: Recommenders pair what was shown with what you did.
3. **Multi-action formulas**: Scoring balances positive engagement (watch time, likes) against swift penalties for skips.
4. **Shortlists over perfection**: Evaluation metrics like Hit@10 measure whether relevant content made it into the top tier of your feed.